# 🤚 Sign Language Translator (Real-Time)

> **Accessibility AI** — Detects hand signs via webcam and translates them into text and speech in real time.

---

## 🖥️ Running This Notebook in VS Code

Follow these steps **once** to set up VS Code with the Jupyter extension:

### 1️⃣ Install the Jupyter extension
1. Open VS Code
2. Press `Ctrl+Shift+X` to open Extensions
3. Search **"Jupyter"** → install the one by **Microsoft**
4. Also install **"Python"** extension if not already installed

### 2️⃣ Open the notebook
1. Press `Ctrl+Shift+E` to open Explorer
2. Navigate to your project folder and click **`sign_language_translator.ipynb`**
3. The notebook opens with cell-by-cell run buttons

### 3️⃣ Select a Python interpreter / kernel
1. Click **"Select Kernel"** (top-right of the notebook)
2. Choose **"Python Environments"**
3. Pick your Python 3.10+ installation (recommended: a virtual env)

### 4️⃣ Run cells
- Click the ▶️ button on a cell, **or** press `Shift+Enter` to run and move to the next
- Run cells **top to bottom** in order (Step 0 → 1 → 2 → 3)
- After Step 0 installs packages: press **"Restart"** in the kernel toolbar, then continue

### 5️⃣ Webcam windows
- Steps 1 and 3 open an **OpenCV window outside VS Code** — click on it to interact
- Keep the OpenCV window **in focus** to use keyboard shortcuts (`s`, `q`, `ENTER`, etc.)

> 💡 **Tip:** If you get import errors after Step 0, click **Restart Kernel** and run from Step 1 again.

---

## 📋 Notebook Outline

| Step | What you do |
|---|---|
| **Step 0** | Install all required libraries |
| **Step 1** | Collect hand-sign images from your webcam |
| **Step 2** | Extract MediaPipe landmarks + train the neural network |
| **Step 3** | Run the real-time translator with text-to-speech |

---

## 🛠️ Tech Stack

| Library | Purpose |
|---|---|
| **MediaPipe** | 21-point hand landmark detection |
| **OpenCV** | Webcam capture + image processing |
| **TensorFlow/Keras** | Neural network classifier |
| **pyttsx3** | Offline text-to-speech |
| **scikit-learn** | Label encoding, train/test split |
| **NumPy / Matplotlib** | Data + plots |

---
## ⚙️ Step 0 — Install Dependencies

Run this cell **once**. Restart the kernel afterwards.

In [ ]:
import sys

!{sys.executable} -m pip install -q \
    opencv-python>=4.8.0 \
    mediapipe>=0.10.0 \
    tensorflow>=2.13.0 \
    numpy>=1.24.0 \
    scikit-learn>=1.3.0 \
    pyttsx3>=2.90 \
    matplotlib>=3.7.0 \
    tqdm>=4.65.0

print('\n✅ All packages installed! Please RESTART the kernel, then continue.')

---
## 📦 Imports & Global Configuration

In [ ]:
import os
import time
import threading
import collections
import warnings
warnings.filterwarnings('ignore')

import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR   = os.getcwd()          # notebook's working directory
DATA_DIR   = os.path.join(BASE_DIR, 'data')
MODEL_DIR  = os.path.join(BASE_DIR, 'model')
MODEL_PATH = os.path.join(MODEL_DIR, 'sign_language_model.h5')
LABEL_PATH = os.path.join(MODEL_DIR, 'label_map.npy')

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Classes to recognise ────────────────────────────────────────────────────
# Default: all 26 English letters + SPACE, DEL, NOTHING
# To use custom words, change this list, e.g.:
# CLASSES = ['Hello', 'Thanks', 'Yes', 'No', 'Help', 'Water', 'Food']
CLASSES = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ') + ['SPACE', 'DEL', 'NOTHING']

IMAGES_PER_CLASS = 300   # images collected per class (300 for better accuracy)

print(f'✅ Setup complete.')
print(f'   Classes ({len(CLASSES)}): {CLASSES}')
print(f'   Data dir  → {DATA_DIR}')
print(f'   Model dir → {MODEL_DIR}')

---
## 📷 Step 1 — Collect Hand-Sign Data

**How it works:**
1. Your webcam opens.
2. For each class the script shows you what sign to make.
3. Press **`s`** to start capturing 100 images for that class.
4. Press **`q`** to stop early / skip.

💡 **Tips for good data:**
- Use good lighting (face a lamp or sit near a window)
- Slightly vary your hand position and distance
- Keep the sign clear and centered

> ⚠️ The webcam window opens **outside** the notebook. Click on it to interact with it.

In [ ]:
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils


def collect_data(classes=CLASSES, images_per_class=IMAGES_PER_CLASS):
    """Open webcam and collect hand-sign images for each class."""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print('❌ Cannot open webcam. Check camera connection.')
        return

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        min_detection_confidence=0.5,
    ) as hands:

        for class_name in classes:
            class_dir = os.path.join(DATA_DIR, class_name)
            os.makedirs(class_dir, exist_ok=True)

            # ── Wait for 's' ──────────────────────────────────────────────
            print(f"\n[INFO] Class: '{class_name}'  →  Press 's' to capture, 'q' to skip/quit")
            while True:
                ret, frame = cap.read()
                if not ret:
                    continue
                frame = cv2.flip(frame, 1)
                cv2.putText(
                    frame,
                    f"Class: {class_name}  |  Press 's' to start",
                    (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2,
                )
                cv2.imshow('Data Collection', frame)
                key = cv2.waitKey(1) & 0xFF
                if key == ord('s'):
                    break
                if key == ord('q'):
                    cap.release()
                    cv2.destroyAllWindows()
                    print('[INFO] Collection stopped.')
                    return

            # ── Capture frames ────────────────────────────────────────────
            count = 0
            while count < images_per_class:
                ret, frame = cap.read()
                if not ret:
                    continue
                frame = cv2.flip(frame, 1)
                rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                results = hands.process(rgb)
                if results.multi_hand_landmarks:
                    for lm in results.multi_hand_landmarks:
                        mp_drawing.draw_landmarks(
                            frame, lm, mp_hands.HAND_CONNECTIONS
                        )

                cv2.putText(
                    frame,
                    f"{class_name}  [{count}/{images_per_class}]",
                    (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 200, 255), 2,
                )
                cv2.imshow('Data Collection', frame)
                cv2.waitKey(1)

                img_path = os.path.join(class_dir, f'{count:04d}.jpg')
                cv2.imwrite(img_path, frame)
                count += 1

            print(f'[INFO] Saved {count} images for "{class_name}"')

    cap.release()
    cv2.destroyAllWindows()
    print('\n✅ Data collection complete! Proceed to Step 2.')


# ▶️ Run data collection
collect_data()

### 🗂️ Check collected data

In [ ]:
print('📁 Images collected per class:\n')
total = 0
for cls in sorted(os.listdir(DATA_DIR)):
    cls_path = os.path.join(DATA_DIR, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        total += count
        bar = '█' * (count // 5)
        print(f'  {cls:10s}  {bar}  {count}')

print(f'\n  Total images: {total}')

---
## 🧠 Step 2 — Feature Extraction + Model Training

### How it works:

```
Image  →  MediaPipe  →  21 hand landmarks × (x, y, z)  →  63 numbers
                                                              ↓
                                              Dense Neural Network
                                                              ↓
                                              Predicted class (A, B, … )
```

**Why landmarks instead of raw pixels?**
- Only 63 numbers vs. 307,200 pixels → **much faster**
- Works regardless of hand position on screen → **position-invariant**
- Runs in real time on a CPU without a GPU

In [ ]:
# ── 2a. Extract Landmarks from Images ──────────────────────────────────────

def extract_landmarks(image_path, hands):
    """Return a (63,) float32 array, or None if no hand detected."""
    img = cv2.imread(image_path)
    if img is None:
        return None
    rgb     = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)
    if not results.multi_hand_landmarks:
        return None
    lm = results.multi_hand_landmarks[0].landmark
    return np.array([[p.x, p.y, p.z] for p in lm], dtype=np.float32).flatten()


def load_dataset():
    X, y = [], []
    classes = sorted(os.listdir(DATA_DIR))

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.3,
    ) as hands:
        for label in tqdm(classes, desc='Extracting landmarks'):
            cls_dir = os.path.join(DATA_DIR, label)
            if not os.path.isdir(cls_dir):
                continue
            for img_file in os.listdir(cls_dir):
                if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                feats = extract_landmarks(os.path.join(cls_dir, img_file), hands)
                if feats is not None:
                    X.append(feats)
                    y.append(label)

    return np.array(X, dtype=np.float32), np.array(y)


print('⏳ Extracting landmarks from all images …')
X, y_raw = load_dataset()
print(f'\n✅ Done!  Samples: {len(X)},  Features per sample: {X.shape[1]}')

In [ ]:
# ── 2b. Encode Labels ───────────────────────────────────────────────────────

le = LabelEncoder()
y  = le.fit_transform(y_raw)
np.save(LABEL_PATH, le.classes_)

print(f'Classes ({len(le.classes_)}): {list(le.classes_)}')
print(f'Label map saved → {LABEL_PATH}')

In [ ]:
# ── 2c. Train / Test Split ──────────────────────────────────────────────────

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} samples  |  Val: {len(X_val)} samples')

In [ ]:
# ── 2d. Build the Model ─────────────────────────────────────────────────────

num_classes = len(le.classes_)

model = models.Sequential([
    layers.Input(shape=(63,)),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(num_classes, activation='softmax'),
], name='sign_language_classifier')

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

In [ ]:
# ── 2e. Train ───────────────────────────────────────────────────────────────

cb_list = [
    callbacks.EarlyStopping(patience=7, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(patience=4, factor=0.5, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=cb_list,
    verbose=1,
)

loss, acc = model.evaluate(X_val, y_val, verbose=0)
print(f'\n✅ Validation Accuracy: {acc * 100:.2f}%')

In [ ]:
# ── 2f. Plot Training History ───────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history['accuracy'],     label='Train Accuracy', color='royalblue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   color='tomato')
axes[0].set_title('Accuracy over Epochs', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train Loss', color='royalblue')
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='tomato')
axes[1].set_title('Loss over Epochs', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(MODEL_DIR, 'training_history.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved → {plot_path}')

In [ ]:
# ── 2g. Save Model ──────────────────────────────────────────────────────────

model.save(MODEL_PATH)
print(f'✅ Model saved → {MODEL_PATH}')
print('   Proceed to Step 3 to run the real-time translator!')

---
## 🎯 Step 3 — Real-Time Sign Language Translator

### How it works:

```
Webcam frame
    → MediaPipe detects 21 hand landmarks
    → Neural network predicts letter (e.g. "A")
    → Smoothed over 10 frames (majority vote)  ← removes jitter
    → Held for 20 stable frames                ← avoids accidental letters
    → Letter added to sentence
    → Press ENTER → pyttsx3 speaks the sentence 🔊
```

### ⌨️ Controls

| Key | Action |
|---|---|
| `SPACE` | Add a space between words |
| `ENTER` | 🔊 Speak the sentence aloud |
| `c` | Clear the sentence |
| `q` | Quit |

> ⚠️ The webcam window opens **outside** the notebook. Click on it to use keyboard controls.

In [ ]:
import pyttsx3

# ── Text-to-speech helper (runs in background thread) ───────────────────────
_tts_engine = pyttsx3.init()
_tts_engine.setProperty('rate', 140)

def speak(text):
    """Speak text without blocking the webcam loop."""
    def _run():
        _tts_engine.say(text)
        _tts_engine.runAndWait()
    threading.Thread(target=_run, daemon=True).start()


# ── UI drawing helper ────────────────────────────────────────────────────────
def draw_ui(frame, prediction, confidence, sentence):
    h, w = frame.shape[:2]

    # Top banner
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, 70), (30, 30, 30), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    cv2.putText(
        frame, f'Sign: {prediction}  ({confidence * 100:.1f}%)',
        (10, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 120), 2,
    )

    # Bottom banner (sentence)
    overlay2 = frame.copy()
    cv2.rectangle(overlay2, (0, h - 85), (w, h), (30, 30, 30), -1)
    cv2.addWeighted(overlay2, 0.6, frame, 0.4, 0, frame)
    cv2.putText(
        frame, f'Sentence: {sentence}',
        (10, h - 45), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 2,
    )
    cv2.putText(
        frame, 'SPACE=space  ENTER=speak  C=clear  Q=quit',
        (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1,
    )
    return frame

print('✅ Helpers ready. Run the next cell to start the translator.')

In [ ]:
# ── Load the trained model ───────────────────────────────────────────────────

if not os.path.exists(MODEL_PATH) or not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(
        '❌ Model not found. Please run Step 2 first to train the model.'
    )

model_rt  = tf.keras.models.load_model(MODEL_PATH)
labels_rt = np.load(LABEL_PATH, allow_pickle=True)
print(f'✅ Model loaded. Classes: {list(labels_rt)}')

# ── Real-time translator ─────────────────────────────────────────────────────

SMOOTHING_FRAMES  = 10    # vote over last N frames
CONFIDENCE_THRESH = 0.80  # minimum confidence
HOLD_FRAMES       = 20    # frames sign must be stable before adding letter

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError('❌ Cannot open webcam.')
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

sentence         = ''
recent_preds     = collections.deque(maxlen=SMOOTHING_FRAMES)
stable_label     = ''
stable_count     = 0
last_added_label = ''
last_added_time  = 0.0

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.5,
) as hands:

    while True:
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.flip(frame, 1)
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        prediction = '—'
        confidence = 0.0

        results = hands.process(rgb)
        if results.multi_hand_landmarks:
            hand_lm = results.multi_hand_landmarks[0]
            mp_drawing.draw_landmarks(
                frame, hand_lm, mp_hands.HAND_CONNECTIONS,
                mp_drawing.DrawingSpec(color=(121, 22, 76),  thickness=2, circle_radius=4),
                mp_drawing.DrawingSpec(color=(250, 44, 250), thickness=2),
            )

            features   = np.array(
                [[lm.x, lm.y, lm.z] for lm in hand_lm.landmark],
                dtype=np.float32
            ).flatten().reshape(1, -1)

            probs      = model_rt.predict(features, verbose=0)[0]
            idx        = int(np.argmax(probs))
            confidence = float(probs[idx])

            if confidence >= CONFIDENCE_THRESH:
                prediction = labels_rt[idx]
                recent_preds.append(prediction)

                if len(recent_preds) == SMOOTHING_FRAMES:
                    counter     = collections.Counter(recent_preds)
                    voted_label = counter.most_common(1)[0][0]

                    if voted_label == stable_label:
                        stable_count += 1
                    else:
                        stable_label = voted_label
                        stable_count = 1

                    now = time.time()
                    if (
                        stable_count >= HOLD_FRAMES
                        and (
                            voted_label != last_added_label
                            or now - last_added_time > 1.5
                        )
                        and voted_label != 'NOTHING'
                    ):
                        if voted_label == 'SPACE':
                            sentence += ' '
                        elif voted_label == 'DEL':
                            sentence = sentence[:-1]
                        else:
                            sentence += voted_label

                        last_added_label = voted_label
                        last_added_time  = now
                        stable_count     = 0
            else:
                recent_preds.clear()
                stable_label = ''
                stable_count = 0

        frame = draw_ui(frame, prediction, confidence, sentence)
        cv2.imshow('🤚 Sign Language Translator', frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == 13:           # ENTER — speak
            if sentence.strip():
                speak(sentence.strip())
        elif key == ord(' '):     # SPACE — add space
            sentence += ' '
        elif key == ord('c'):     # C — clear
            sentence = ''

cap.release()
cv2.destroyAllWindows()
print('✅ Translator closed.')

---
## 🔧 Customise — Use Your Own Words Instead of Letters

Change the `CLASSES` list in the **Imports & Configuration** cell to any words you want, for example:

```python
CLASSES = ['Hello', 'Thanks', 'Yes', 'No', 'Help', 'Water', 'Food', 'NOTHING']
```

Then re-run **Step 1** (collect new images) and **Step 2** (re-train) — done! 🎉

---
## ❓ Troubleshooting

| Problem | Fix |
|---|---|
| `Cannot open webcam` | Check camera is connected; try `VideoCapture(1)` |
| Low accuracy (<80%) | Collect 200+ images with varied hand positions |
| Prediction is jittery | Increase `SMOOTHING_FRAMES` to 15–20 |
| No sound on ENTER | On Linux: `sudo apt install espeak` |
| `Model not found` | Run Step 2 before Step 3 |

---
## 👨‍💻 Author

**Ali Muddassar** — CS Student · Aspiring Data Scientist  
[GitHub](https://github.com/Ali-Muddassar) · [LinkedIn](https://www.linkedin.com/in/ali-muddassar-17466a3b7)